# Module `algorithms/neighborhood.py`

Ce notebook presente la fondation partagee par toutes les metaheuristiques VRP-CDR : la representation `VRPSolution`, les operateurs de voisinage deterministes (`relocate_inter`, `swap_inter`, `two_opt`), la recherche locale composite `local_search`, et les tirages aleatoires (`random_*`) utilises par le recuit simule.

**Regle transverse du projet** : tout nouvel operateur de voisinage doit etre ajoute ici, et non duplique dans chaque algorithme.


In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent.parent / "src"))

from cesipath import GraphGenerationConfig, GraphGenerator
from cesipath.algorithms import (
    VRPSolution,
    is_feasible,
    local_search,
    random_neighbor,
    relocate_inter,
    route_cost,
    route_load,
    swap_inter,
    total_cost,
    two_opt,
    two_opt_intra,
)
from cesipath.algorithms.grasp import greedy_randomized_construction
from cesipath.solver_input import build_static_solver_input
import random


## 1. Instance et solution initiale

On fabrique une petite instance puis une solution admissible via le constructeur glouton de GRASP. Cela donne une base concrete sur laquelle manipuler les operateurs.


In [2]:
cfg = GraphGenerationConfig(node_count=12, seed=7)
instance = GraphGenerator(cfg).generate()
si = build_static_solver_input(instance)

rng = random.Random(0)
routes = greedy_randomized_construction(si, rcl_alpha=0.0, rng=rng)

print(f"{len(routes)} sous-tournees, cout initial = {total_cost(routes, si.cost_matrix):.2f}")
for idx, r in enumerate(routes):
    print(f"  r{idx + 1} : {r}  charge={route_load(r, si.demands)}/{si.vehicle_capacity}")


2 sous-tournees, cout initial = 1002.17
  r1 : [0, 2, 11, 1, 7, 3, 5, 4, 9, 0]  charge=40/40
  r2 : [0, 6, 8, 10, 0]  charge=15/40


## 2. Representation `VRPSolution`

`VRPSolution` est une `dataclass` qui encapsule la liste de routes et le cout total. Les operateurs renvoient des `Routes` (`list[list[int]]`) et c'est la couche algorithme qui construit l'objet final en sortie. La propriete `route_count` est un raccourci sur `len(routes)`.


In [3]:
sol = VRPSolution(routes=routes, total_cost=round(total_cost(routes, si.cost_matrix), 2))
print(sol)
print("nombre de tournees :", sol.route_count)


VRPSolution(routes=[[0, 2, 11, 1, 7, 3, 5, 4, 9, 0], [0, 6, 8, 10, 0]], total_cost=1002.17)
nombre de tournees : 2


## 3. Cout de tournee et charge

- `route_cost` somme les aretes consecutives d'une route (depot -> clients -> depot).
- `route_load` somme les demandes **clients** (on exclut les depots aux extremites).
- `is_feasible` verifie structure + capacite mais **pas** la validite metier (arcs interdits, couverture totale).


In [4]:
r0 = routes[0]
print("route r1 :", r0)
print("cout r1  :", round(route_cost(r0, si.cost_matrix), 2))
print("charge r1:", route_load(r0, si.demands))
print("admissible ?", is_feasible(routes, si.demands, si.vehicle_capacity))


route r1 : [0, 2, 11, 1, 7, 3, 5, 4, 9, 0]
cout r1  : 364.35
charge r1: 40
admissible ? True


## 4. `two_opt` intra-tournee

2-opt supprime deux aretes non adjacentes et les reconnecte en inversant le segment intermediaire. Best-improvement : on applique le meilleur gain trouve, et on recommence jusqu'a stabilite.

`two_opt_intra` applique `two_opt` a chaque tournee de la solution.


In [5]:
before = total_cost(routes, si.cost_matrix)
after_routes = two_opt_intra(routes, si.cost_matrix)
after = total_cost(after_routes, si.cost_matrix)
print(f"avant 2-opt : {before:.2f}")
print(f"apres 2-opt : {after:.2f}   (gain {before - after:.2f})")


avant 2-opt : 1002.17
apres 2-opt : 906.00   (gain 96.17)


## 5. `relocate_inter` : deplacer un client entre tournees

Best-improvement : on teste chaque (client, position cible) sous contrainte de capacite, on applique le meilleur mouvement strictement ameliorant puis on itere. L'operateur retourne une copie et **nettoie les tournees vides** (`_prune_empty_routes`).


In [6]:
before = total_cost(routes, si.cost_matrix)
relocated = relocate_inter(routes, si.cost_matrix, si.demands, si.vehicle_capacity)
after = total_cost(relocated, si.cost_matrix)
print(f"avant relocate : {before:.2f}   ({len(routes)} tournees)")
print(f"apres relocate : {after:.2f}   ({len(relocated)} tournees)")


avant relocate : 1002.17   (2 tournees)
apres relocate : 769.86   (2 tournees)


## 6. `swap_inter` : echanger deux clients entre tournees

Meme logique best-improvement. Contrairement a `relocate_inter`, `swap_inter` conserve le nombre de tournees puisque chaque route perd un client et en gagne un autre.


In [7]:
swapped = swap_inter(relocated, si.cost_matrix, si.demands, si.vehicle_capacity)
print(f"apres swap     : {total_cost(swapped, si.cost_matrix):.2f}   ({len(swapped)} tournees)")


apres swap     : 769.86   (2 tournees)


## 7. `local_search` : composition jusqu'a stabilite

`local_search` chaine `relocate_inter -> swap_inter -> two_opt_intra` et recommence tant que le cout strict baisse. C'est la routine de descente utilisee par GRASP, par le schema memetique de l'algorithme genetique et en passe finale des recuits / tabu.


In [8]:
ls_routes = local_search(routes, si.cost_matrix, si.demands, si.vehicle_capacity)
print(f"cout initial    : {total_cost(routes, si.cost_matrix):.2f}")
print(f"apres local_search : {total_cost(ls_routes, si.cost_matrix):.2f}")
print(f"nombre de tournees : {len(ls_routes)}")


cout initial    : 1002.17
apres local_search : 769.86
nombre de tournees : 2


## 8. Voisins aleatoires

Le recuit simule n'explore pas tout le voisinage. Il tire un mouvement au hasard via `random_neighbor`, qui choisit uniformement entre `relocate`, `swap`, et `two_opt` aleatoires.

Chaque `random_*` renvoie `None` si le tirage ne produit pas de voisin admissible ; `random_neighbor` retombe alors sur une copie de la solution courante pour garantir un retour valide.


In [9]:
rng = random.Random(42)
for step in range(6):
    neighbor = random_neighbor(ls_routes, si.demands, si.vehicle_capacity, rng)
    c = total_cost(neighbor, si.cost_matrix)
    print(f"voisin {step + 1} : cout={c:.2f} ; {len(neighbor)} tournees")


voisin 1 : cout=830.21 ; 2 tournees
voisin 2 : cout=919.04 ; 2 tournees
voisin 3 : cout=838.74 ; 2 tournees
voisin 4 : cout=935.86 ; 2 tournees
voisin 5 : cout=785.67 ; 2 tournees
voisin 6 : cout=836.95 ; 2 tournees


## 9. Reproductibilite

Les operateurs deterministes (`relocate_inter`, `swap_inter`, `two_opt`) sont strictement reproductibles : meme entree => meme sortie. Les operateurs aleatoires dependent d'une `random.Random` passee explicitement : deux executions avec la meme graine produisent la meme trajectoire, ce qui facilite le debug des metaheuristiques.


In [10]:
def trajectory(seed: int) -> list[float]:
    rng = random.Random(seed)
    s = [route[:] for route in ls_routes]
    out = []
    for _ in range(5):
        s = random_neighbor(s, si.demands, si.vehicle_capacity, rng)
        out.append(round(total_cost(s, si.cost_matrix), 2))
    return out

print("seed=1 :", trajectory(1))
print("seed=1 :", trajectory(1))
print("seed=2 :", trajectory(2))


seed=1 : [879.28, 996.22, 1128.58, 1063.31, 1063.31]
seed=1 : [879.28, 996.22, 1128.58, 1063.31, 1063.31]
seed=2 : [805.17, 921.73, 913.2, 972.24, 1032.62]
